# Fine-tuning LLM Tiếng Việt bằng QLoRA/LoRA trên Google Colab

Model: **Qwen2.5-1.5B-Instruct**  
Framework: Hugging Face Transformers + TRL + PEFT

> Trước khi chạy: vào **Runtime -> Change runtime type -> GPU** (chọn T4 hoặc cao hơn).

In [38]:
!pip install -q -U \
    transformers \
    datasets \
    accelerate \
    peft \
    trl \
    bitsandbytes \
    sentencepiece


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 578, in _determine_conflicts
    return check_install_conflicts(to_install)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/operations/check.py", line 101, in check_install_conflicts
    package_set, _ = create_package_set_from_installed()
              

Nếu Colab yêu cầu **restart runtime** sau khi cài, hãy chọn `Runtime -> Restart session` rồi chạy tiếp từ cell dưới.

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("Không có GPU. Hãy chọn Runtime -> Change runtime type -> GPU")


PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
CUDA version: 12.8


In [ ]:
import os
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

from peft import (
    LoraConfig,
    PeftModel
)

# Lưu ý: SFTConfig thay thế cho TrainingArguments trong các bản TRL mới.
# dataset_text_field / max_length / packing giờ là thuộc tính của SFTConfig,
# không còn truyền trực tiếp vào SFTTrainer nữa.
from trl import SFTTrainer, SFTConfig


## Dataset tiếng Việt mẫu

Bạn có thể thay thế danh sách `data` bên dưới bằng dữ liệu của riêng bạn (càng nhiều mẫu càng tốt, tối thiểu vài trăm mẫu để có kết quả ổn).

In [ ]:
data = [
    {
        "instruction": "Giải thích trí tuệ nhân tạo là gì.",
        "response":
        "Trí tuệ nhân tạo, hay AI, là lĩnh vực nghiên cứu các phương pháp "
        "giúp máy tính thực hiện những nhiệm vụ thường cần đến trí thông minh "
        "của con người như nhận dạng hình ảnh, hiểu ngôn ngữ, suy luận và ra quyết định."
    },
    {
        "instruction": "Machine Learning là gì?",
        "response":
        "Machine Learning là một nhánh của trí tuệ nhân tạo, trong đó máy tính "
        "học các mẫu từ dữ liệu để dự đoán hoặc đưa ra quyết định mà không cần "
        "lập trình tường minh cho từng trường hợp."
    },
    {
        "instruction": "Deep Learning khác Machine Learning như thế nào?",
        "response":
        "Deep Learning là một nhánh của Machine Learning sử dụng mạng neural nhiều tầng. "
        "Deep Learning có khả năng tự học biểu diễn đặc trưng từ dữ liệu thô và đặc biệt "
        "hiệu quả đối với hình ảnh, âm thanh và ngôn ngữ tự nhiên."
    },
    {
        "instruction": "Giải thích mạng neural nhân tạo.",
        "response":
        "Mạng neural nhân tạo là mô hình tính toán được lấy cảm hứng từ hệ thần kinh sinh học. "
        "Mạng gồm nhiều neuron nhân tạo liên kết với nhau thông qua các trọng số và thường "
        "được tổ chức thành lớp đầu vào, các lớp ẩn và lớp đầu ra."
    },
    {
        "instruction": "Gradient Descent là gì?",
        "response":
        "Gradient Descent là thuật toán tối ưu dùng để tìm giá trị tham số làm giảm hàm mất mát. "
        "Tại mỗi bước, tham số được cập nhật theo hướng ngược với gradient của hàm mất mát."
    },
    {
        "instruction": "Transformer là gì?",
        "response":
        "Transformer là kiến trúc mạng neural dựa chủ yếu trên cơ chế attention. "
        "Transformer cho phép mô hình xác định mức độ liên quan giữa các từ trong một chuỗi "
        "và là nền tảng của nhiều mô hình ngôn ngữ lớn hiện đại."
    },
    {
        "instruction": "LLM là gì?",
        "response":
        "LLM là viết tắt của Large Language Model, tức mô hình ngôn ngữ lớn. "
        "LLM được huấn luyện trên lượng dữ liệu văn bản rất lớn để học cấu trúc và quy luật "
        "của ngôn ngữ, từ đó có thể sinh văn bản, trả lời câu hỏi và thực hiện nhiều nhiệm vụ NLP."
    },
    {
        "instruction": "Fine-tuning là gì?",
        "response":
        "Fine-tuning là quá trình tiếp tục huấn luyện một mô hình đã được pre-train "
        "trên một tập dữ liệu chuyên biệt nhằm giúp mô hình thích nghi với một nhiệm vụ "
        "hoặc lĩnh vực cụ thể."
    },
    {
        "instruction": "LoRA là gì?",
        "response":
        "LoRA là kỹ thuật fine-tuning hiệu quả tham số. Thay vì cập nhật toàn bộ trọng số "
        "của mô hình lớn, LoRA chỉ học một số ma trận hạng thấp được bổ sung vào các lớp "
        "của mô hình, giúp giảm đáng kể bộ nhớ và chi phí huấn luyện."
    },
    {
        "instruction": "QLoRA khác LoRA ở điểm nào?",
        "response":
        "QLoRA kết hợp lượng tử hóa mô hình nền, thường xuống 4-bit, với LoRA. "
        "Nhờ đó mô hình lớn có thể được fine-tune với lượng GPU memory thấp hơn "
        "trong khi vẫn giữ hiệu quả tốt."
    },
    {
        "instruction": "Attention trong Transformer hoạt động như thế nào?",
        "response":
        "Cơ chế attention cho phép mô hình gán trọng số khác nhau cho từng từ trong chuỗi "
        "đầu vào tùy theo mức độ liên quan của chúng với từ đang được xử lý, giúp mô hình "
        "nắm bắt ngữ cảnh dài và mối quan hệ giữa các từ ở xa nhau."
    },
    {
        "instruction": "Overfitting là gì và làm sao để giảm thiểu?",
        "response":
        "Overfitting xảy ra khi mô hình học quá khớp với dữ liệu huấn luyện, dẫn đến "
        "hiệu suất kém trên dữ liệu mới. Có thể giảm thiểu bằng cách dùng nhiều dữ liệu hơn, "
        "regularization, dropout, hoặc dừng huấn luyện sớm (early stopping)."
    },
]

dataset = Dataset.from_list(data)

print(dataset)
print(dataset[0])


Dataset({
    features: ['instruction', 'response'],
    num_rows: 12
})
{'instruction': 'Giải thích trí tuệ nhân tạo là gì.', 'response': 'Trí tuệ nhân tạo, hay AI, là lĩnh vực nghiên cứu các phương pháp giúp máy tính thực hiện những nhiệm vụ thường cần đến trí thông minh của con người như nhận dạng hình ảnh, hiểu ngôn ngữ, suy luận và ra quyết định.'}


In [ ]:
dataset = dataset.train_test_split(
    test_size=0.2,
    seed=42
)

train_dataset = dataset["train"]
test_dataset = dataset["test"]

print("Train:", len(train_dataset))
print("Test :", len(test_dataset))


Train: 9
Test : 3


In [ ]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

print("Model:", model_name)


Model: Qwen/Qwen2.5-1.5B-Instruct


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("Vocab size:", len(tokenizer))


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Vocab size: 151665


In [ ]:
SYSTEM_PROMPT = (
    "Bạn là trợ lý AI sử dụng tiếng Việt. "
    "Hãy trả lời chính xác, rõ ràng và dễ hiểu."
)


def format_example(example):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["response"]},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    return {"text": text}


train_dataset = train_dataset.map(format_example)
test_dataset = test_dataset.map(format_example)

print(train_dataset[0]["text"])


Map:   0%|          | 0/9 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

<|im_start|>system
Bạn là trợ lý AI sử dụng tiếng Việt. Hãy trả lời chính xác, rõ ràng và dễ hiểu.<|im_end|>
<|im_start|>user
QLoRA khác LoRA ở điểm nào?<|im_end|>
<|im_start|>assistant
QLoRA kết hợp lượng tử hóa mô hình nền, thường xuống 4-bit, với LoRA. Nhờ đó mô hình lớn có thể được fine-tune với lượng GPU memory thấp hơn trong khi vẫn giữ hiệu quả tốt.<|im_end|>



## Cấu hình QLoRA 4-bit và load model

In [ ]:
# QUAN TRỌNG: dùng bfloat16 xuyên suốt (thay vì float16).
#
# Lý do: lỗi
#   NotImplementedError: "_amp_foreach_non_finite_check_and_unscale_cuda"
#   not implemented for 'BFloat16'
# xảy ra vì chế độ fp16=True bắt buộc dùng torch.cuda.amp.GradScaler, và
# GradScaler KHÔNG hỗ trợ bất kỳ tensor bfloat16 nào (Qwen2.5 vốn có vài
# phần luôn ở bfloat16 dù ta có ép torch_dtype=float16 hay không). Chỉ cần
# lọt một layer bfloat16 là toàn bộ training crash.
#
# Cách chắc chắn nhất: chuyển toàn bộ sang bfloat16. Khi dùng bf16=True,
# Trainer KHÔNG dùng GradScaler nữa -> lỗi này biến mất hoàn toàn.
# GPU T4 vẫn chạy được bfloat16 (không có Tensor Core native cho bf16 như
# GPU đời mới, nên hơi chậm hơn chút, nhưng chạy đúng và ổn định).

compute_dtype = torch.bfloat16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,

    # NormalFloat4 - thường dùng trong QLoRA
    bnb_4bit_quant_type="nf4",

    # Double quantization giúp giảm memory thêm
    bnb_4bit_use_double_quant=True,

    bnb_4bit_compute_dtype=compute_dtype
)


In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    torch_dtype=compute_dtype,
    device_map="auto",
    trust_remote_code=True
)

model.config.use_cache = False

print("Đã load model.")
print("Model dtype:", model.dtype)


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Đã load model.
Model dtype: torch.bfloat16


In [ ]:
from peft import prepare_model_for_kbit_training

# Bước chuẩn khi dùng QLoRA: đưa layernorm về fp32, bật input require_grad,
# chuẩn bị model tương thích với gradient checkpointing. Thiếu bước này
# cũng là nguyên nhân phổ biến gây lỗi dtype/NaN khi train.
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True
)


In [ ]:
peft_config = LoraConfig(

    # rank của LoRA
    r=16,

    # hệ số scale
    lora_alpha=32,

    lora_dropout=0.05,

    bias="none",

    task_type="CAUSAL_LM",

    # Các layer attention + MLP phổ biến của Qwen/Llama
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

print(peft_config)


LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.20.0', base_model_name_or_path=None, revision=None, inference_mode=False, r=16, target_modules={'k_proj', 'gate_proj', 'o_proj', 'v_proj', 'up_proj', 'down_proj', 'q_proj'}, exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, velora_config=None, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, monteclora_config=None, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, ensure_weight_tying=False)


## Cấu hình training (`SFTConfig`)

**Lưu ý quan trọng:** trong các bản TRL mới (>=0.19–0.20 trở lên), `dataset_text_field`, `max_length` (đổi tên từ `max_seq_length`) và `packing` là thuộc tính của `SFTConfig`, không còn truyền trực tiếp vào `SFTTrainer` nữa. Đây là nguyên nhân phổ biến nhất gây lỗi `TypeError` khi chạy các đoạn code fine-tuning cũ.

In [ ]:
training_args = SFTConfig(

    output_dir="./qwen-vietnamese-lora",

    # số epoch
    num_train_epochs=3,

    # batch nhỏ phù hợp Colab
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    # batch hiệu dụng = 1 x 8
    gradient_accumulation_steps=8,

    # learning rate cho LoRA thường lớn hơn full fine-tuning
    learning_rate=2e-4,

    # optimizer tốt cho QLoRA
    optim="paged_adamw_8bit",

    # log
    logging_steps=1,

    # save
    save_strategy="steps",
    save_steps=20,
    save_total_limit=2,

    # eval
    eval_strategy="epoch",

    # precision: dùng bf16 (không dùng fp16) để tránh lỗi GradScaler với
    # BFloat16 (xem ghi chú ở cell load model phía trên)
    fp16=False,
    bf16=True,

    # giảm memory
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    # scheduler
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.01,

    # Hugging Face reporting
    report_to="none",

    # --- các tham số dành riêng cho SFT (trước đây truyền cho SFTTrainer) ---
    dataset_text_field="text",
    max_length=512,
    packing=False,
)


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    peft_config=peft_config,
    args=training_args,

    # tokenizer= đã bị loại bỏ khỏi TRL, dùng processing_class thay thế
    processing_class=tokenizer,
)


Adding EOS to train dataset:   0%|          | 0/9 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/9 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/9 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/9 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/9 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/3 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/3 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/3 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/3 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/3 [00:00<?, ? examples/s]

In [ ]:
trainer.model.print_trainable_parameters()


trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


## Bắt đầu fine-tuning

Với dataset mẫu (12 câu) và cấu hình trên, quá trình này chạy trong vài phút trên GPU T4 miễn phí của Colab.

In [ ]:
trainer.train()


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,2.725101,2.255900,1.749714,882.000000,0.596825
2,1.852763,1.615248,1.579232,1764.000000,0.690196
3,1.781411,1.513555,1.529020,2646.000000,0.686275


TrainOutput(global_step=6, training_loss=2.349947154521942, metrics={'train_runtime': 28.5045, 'train_samples_per_second': 0.947, 'train_steps_per_second': 0.21, 'total_flos': 21096114149376.0, 'train_loss': 2.349947154521942, 'epoch': 3.0})

In [ ]:
save_path = "./qwen-vietnamese-lora-final"

trainer.model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print("Đã lưu model tại:", save_path)


Đã lưu model tại: ./qwen-vietnamese-lora-final


## Kiểm tra model sau fine-tuning

In [ ]:
model.eval()


def build_prompt(question: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )


question = "Giải thích LoRA bằng ngôn ngữ đơn giản."

prompt = build_prompt(question)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.pad_token_id,
    )

answer = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("Câu hỏi:")
print(question)

print("\nTrả lời:")
print(answer)


Câu hỏi:
Giải thích LoRA bằng ngôn ngữ đơn giản.

Trả lời:
LoRA (Low-Rank Adaptability) là một công nghệ để cải thiện hiệu suất của mô hình AI mà không cần nhiều dữ liệu đầu vào lớn. Nó giúp mô hình học được từ ít dữ liệu hơn, nhưng vẫn có thể xử lý được các tác vụ phức tạp với số lượng dữ liệu nhỏ.


In [ ]:
def chat(question: str) -> str:
    prompt = build_prompt(question)

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return response


In [ ]:
print(
    chat(
        "Tại sao Transformer phù hợp với xử lý ngôn ngữ tự nhiên?"
    )
)


Transformer là một loại mô hình neural network được thiết kế để xử lý thông tin ở nhiều chiều, trong đó dữ liệu có thể di chuyển giữa các khối của mô hình mà không cần phải di chuyển toàn bộ thông tin. Mô hình này đặc biệt hiệu quả trong việc phân tích và dự đoán từ ngữ, do nó không có cấu trúc truyền thống như LSTM hoặc RNN.

Trong ngữ pháp, Transformer sử dụng trọng số cho mỗi cột, giúp giảm bớt sự phụ thuộc giữa các chuỗi dữ liệu. Điều này làm giảm độ phức tạp về mặt tính toán và giúp mô hình nhanh chóng hơn trong việc xử lý dữ liệu lớn.


#XÀI LOCAL LLM CHO RAG

In [ ]:
USE_LOCAL_MODEL = True  # đổi thành False nếu muốn dùng Gemini API thay vì model cục bộ

In [ ]:
if USE_LOCAL_MODEL:
    import os
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    from peft import PeftModel

    LOCAL_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
    LOCAL_ADAPTER_PATH = "qwen-vietnamese-lora-final"  # đổi nếu bạn đặt tên thư mục khác

    # Dùng bfloat16 xuyên suốt để tránh lỗi GradScaler/BFloat16 thường gặp khi
    # trộn lẫn fp16 và bf16 (lưu ý: đây là suy luận/inference nên không cần
    # GradScaler, nhưng vẫn nên đồng nhất dtype để tránh cảnh báo/lỗi không cần thiết).
    compute_dtype = torch.bfloat16

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=compute_dtype,
    )

    local_tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_NAME, trust_remote_code=True)
    if local_tokenizer.pad_token is None:
        local_tokenizer.pad_token = local_tokenizer.eos_token

    print(f"[Local LLM] Đang tải model gốc: {LOCAL_MODEL_NAME} ...")
    local_base_model = AutoModelForCausalLM.from_pretrained(
        LOCAL_MODEL_NAME,
        quantization_config=bnb_config,
        torch_dtype=compute_dtype,
        device_map="auto",
        trust_remote_code=True,
    )
    # Đây là suy luận (inference), không phải training, nên bật lại cache để sinh
    # văn bản nhanh hơn (khác với lúc fine-tune phải tắt use_cache).
    local_base_model.config.use_cache = True

    if os.path.isdir(LOCAL_ADAPTER_PATH):
        print(f"[Local LLM] Tìm thấy adapter LoRA tại '{LOCAL_ADAPTER_PATH}', đang gắn vào model gốc...")
        local_model = PeftModel.from_pretrained(local_base_model, LOCAL_ADAPTER_PATH)
    else:
        print(
            f"[Local LLM] Không thấy thư mục adapter '{LOCAL_ADAPTER_PATH}'. "
            "Dùng thẳng model Qwen2.5-1.5B-Instruct gốc (khuyến nghị cho RAG)."
        )
        local_model = local_base_model

    local_model.eval()
    print("[Local LLM] Sẵn sàng.")
else:
    local_model = None
    local_tokenizer = None


[Local LLM] Đang tải model gốc: Qwen/Qwen2.5-1.5B-Instruct ...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[Local LLM] Tìm thấy adapter LoRA tại 'qwen-vietnamese-lora-final', đang gắn vào model gốc...
[Local LLM] Sẵn sàng.


In [ ]:
import torch

LOCAL_SYSTEM_PROMPT = (
    "Bạn là trợ lý AI sử dụng tiếng Việt. "
    "Hãy trả lời chính xác, rõ ràng và dễ hiểu."
)


def local_generate(prompt, max_new_tokens=400, temperature=0.3):
    """Sinh câu trả lời bằng model Qwen2.5 cục bộ (dùng khi USE_LOCAL_MODEL=True)."""
    messages = [
        {"role": "system", "content": LOCAL_SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    text = local_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = local_tokenizer(text, return_tensors="pt").to(local_model.device)

    with torch.no_grad():
        outputs = local_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1,
            pad_token_id=local_tokenizer.pad_token_id,
        )

    return local_tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )


print("[Local LLM] Đã định nghĩa hàm local_generate().")

[Local LLM] Đã định nghĩa hàm local_generate().


In [ ]:
import os
from google.colab import files

os.makedirs("documents", exist_ok=True)
uploaded = files.upload()

for fname in uploaded.keys():
    dest = os.path.join("documents", fname)
    os.rename(fname, dest)

print("\nĐã tải lên và lưu vào thư mục 'documents/':")
for fname in uploaded.keys():
    print(" -", fname)

Saving Information_Retrieval_MekongDigitalMuseum.pdf to Information_Retrieval_MekongDigitalMuseum.pdf

Đã tải lên và lưu vào thư mục 'documents/':
 - Information_Retrieval_MekongDigitalMuseum.pdf


In [ ]:
!pip install -q sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 68.4 MB/s eta 0:00:00


In [ ]:
print(local_generate("Xin chào. Hãy trả lời một câu ngắn bằng tiếng Việt."))

Xin chào! Tôi có thể giúp gì cho bạn?


In [ ]:
import glob
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss

try:
    from pypdf import PdfReader
except ImportError:
    PdfReader = None

# ---------------- Cấu hình ----------------
# Model embedding chạy CỤC BỘ, MIỄN PHÍ (đa ngôn ngữ, hỗ trợ tốt tiếng Việt).
EMBEDDING_MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"

DOCUMENTS_FOLDER = "/content/documents"
CHUNK_SIZE = 500      # số từ mỗi đoạn
CHUNK_OVERLAP = 80    # số từ chồng lấn giữa 2 đoạn liên tiếp
TOP_K = 3             # số đoạn liên quan nhất lấy ra để hỏi LLM


# ---------------- Đọc dữ liệu ----------------
def read_txt_file(path):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()

def read_pdf_file(path):
    if PdfReader is None:
        raise RuntimeError("Chưa cài pypdf. Chạy: !pip install pypdf")
    reader = PdfReader(path)
    return "\n".join(page.extract_text() or "" for page in reader.pages)

def load_documents(folder):
    docs = []
    paths = glob.glob(os.path.join(folder, "**", "*.*"), recursive=True)
    for path in paths:
        ext = os.path.splitext(path)[1].lower()
        try:
            if ext in (".txt", ".md"):
                text = read_txt_file(path)
            elif ext == ".pdf":
                text = read_pdf_file(path)
            else:
                continue
            if text.strip():
                docs.append({"source": os.path.basename(path), "text": text})
        except Exception as e:
            print(f"[Cảnh báo] Không đọc được file {path}: {e}")
    return docs


# ---------------- Chia đoạn (chunking) ----------------
def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    words = text.split()
    if not words:
        return []
    chunks = []
    step = max(chunk_size - overlap, 1)
    for start in range(0, len(words), step):
        chunk_words = words[start:start + chunk_size]
        if chunk_words:
            chunks.append(" ".join(chunk_words))
        if start + chunk_size >= len(words):
            break
    return chunks


# ---------------- Pipeline RAG ----------------
class RAGPipeline:
    def __init__(self, embedding_model_name=EMBEDDING_MODEL_NAME):
        print(f"[Khởi tạo] Đang tải model embedding cục bộ: {embedding_model_name} ...")
        self.embedder = SentenceTransformer(embedding_model_name)
        self.index = None
        self.chunk_texts = []
        self.chunk_sources = []

    def build_index(self, folder):
        docs = load_documents(folder)
        if not docs:
            raise RuntimeError(
                f"Không tìm thấy file .txt/.pdf nào trong thư mục '{folder}'. "
                f"Hãy chạy lại Bước 3 để tải tài liệu lên."
            )

        print(f"[Nạp dữ liệu] Đọc được {len(docs)} tài liệu. Đang chia đoạn (chunking)...")
        for doc in docs:
            for chunk in chunk_text(doc["text"]):
                self.chunk_texts.append(chunk)
                self.chunk_sources.append(doc["source"])

        print(f"[Nạp dữ liệu] Tổng cộng {len(self.chunk_texts)} đoạn (chunks).")
        print("[Embedding] Đang tính vector cho từng đoạn (chạy cục bộ, miễn phí)...")

        embeddings = self.embedder.encode(
            self.chunk_texts,
            batch_size=32,
            show_progress_bar=True,
            normalize_embeddings=True,
        )
        embeddings = np.asarray(embeddings, dtype="float32")

        dim = embeddings.shape[1]
        self.index = faiss.IndexFlatIP(dim)
        self.index.add(embeddings)

        print("[Hoàn tất] Chỉ mục FAISS đã sẵn sàng.")

    def retrieve(self, query, top_k=TOP_K):
        query_vec = self.embedder.encode([query], normalize_embeddings=True)
        query_vec = np.asarray(query_vec, dtype="float32")
        scores, indices = self.index.search(query_vec, top_k)
        results = []
        for score, idx in zip(scores[0], indices[0]):
            if idx == -1:
                continue
            results.append({
                "text": self.chunk_texts[idx],
                "source": self.chunk_sources[idx],
                "score": float(score),
            })
        return results

    def generate_answer(self, query, contexts):
        context_block = "\n\n".join(
            f"[Nguon: {c['source']}]\n{c['text']}" for c in contexts
        )
        prompt = f"""Bạn là một trợ lý trả lời câu hỏi dựa HOÀN TOÀN trên các đoạn
văn bản ngữ cảnh được cung cấp bên dưới. Nếu ngữ cảnh không chứa đủ thông tin
để trả lời, hãy nói rõ là bạn không tìm thấy thông tin đó trong tài liệu,
KHÔNG được tự bịa ra thông tin.

### NGỮ CẢNH:
{context_block}

### CÂU HỎI:
{query}

### TRẢ LỜI (ngắn gọn, chính xác, dựa trên ngữ cảnh trên):"""

        # Model 1.5B khá nhỏ nên giữ prompt ngắn gọn, temperature thấp để bám
        # sát ngữ cảnh, hạn chế bịa thông tin.
        return local_generate(prompt, max_new_tokens=400, temperature=0.3)

    def ask(self, query, top_k=TOP_K):
        contexts = self.retrieve(query, top_k=top_k)
        if not contexts:
            return "Không tìm thấy đoạn văn bản nào liên quan trong tài liệu.", []
        answer = self.generate_answer(query, contexts)
        return answer, contexts

print("Đã định nghĩa xong pipeline RAGPipeline.")


Đã định nghĩa xong pipeline RAGPipeline.


In [ ]:
rag = RAGPipeline()
rag.build_index(DOCUMENTS_FOLDER)

In [ ]:
query = "What is AIM-MD?"

answer, contexts = rag.ask(query)

print("--- TRẢ LỜI ---")
print(answer)

print("\n--- NGUỒN THAM KHẢO (retrieval) ---")
for i, c in enumerate(contexts, 1):
    preview = c["text"][:150].replace("\n", " ")
    print(f"[{i}] {c['source']} (độ tương đồng={c['score']:.3f}): {preview}...")
